# SPARK Tutorial 2026
## Brain Tumor Segmentation with U-Net (PyTorch)

<center><img src="https://www.mayoclinic.org/-/media/kcms/gbs/patient-consumer/images/2014/10/30/15/17/mcdc7_brain_cancer-8col.jpg" alt="Brain MRI" style='width:350px;'></center>

**Image source**: Mayo Clinic

## About This Tutorial

This hands-on tutorial walks you through **brain tumor segmentation** using the **U-Net** architecture implemented in **PyTorch**. We will:

1. **Understand** the clinical motivation for automated brain tumor segmentation
2. **Learn** key image segmentation concepts from the ground up
3. **Explore** evaluation metrics (Dice, IoU, F1, Precision, Recall)
4. **Deep-dive** into the U-Net architecture layer by layer
5. **Survey** state-of-the-art models: nnU-Net, Swin UNETR, MedSAM, TransBTS, and Attention U-Net
6. **Implement** a complete segmentation pipeline on the LGG MRI dataset (Kaggle)
7. **Evaluate** and **visualize** model predictions

> **Environment**: This notebook is designed to run on **Kaggle** with GPU acceleration.
> **Dataset**: [LGG MRI Segmentation](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation) (Brain MRI images with manual FLAIR abnormality segmentation masks).
> **Framework**: PyTorch + segmentation-models-pytorch (smp)

**Note**: A companion TensorFlow/Keras version of this tutorial is also available (`SPARK_BrainTumorSeg2026.ipynb`).

**References**:
- Lee et al., *Data in Brief*, 2024. [DOI:10.1016/j.dib.2024.111159](https://doi.org/10.1016/j.dib.2024.111159)
- Ronneberger et al., *U-Net: Convolutional Networks for Biomedical Image Segmentation*, MICCAI 2015. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)

# <p style="background-color:#2c3e50;color:white;font-size:22px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Table of Contents</p>

1. [Part 1 — Background & Motivation](#part1)
    - [1.1 What is a Brain Tumor?](#sec1_1)
    - [1.2 Medical Imaging & MRI](#sec1_2)
    - [1.3 What is Image Segmentation?](#sec1_3)
    - [1.4 Why Segmentation for Brain Tumors?](#sec1_4)
2. [Part 2 — Segmentation Metrics Deep Dive](#part2)
    - [2.1 Dice Coefficient / F1 Score](#sec2_1)
    - [2.2 Intersection over Union (IoU / Jaccard)](#sec2_2)
    - [2.3 Precision, Recall & Pixel Accuracy](#sec2_3)
3. [Part 3 — U-Net Architecture Deep Dive](#part3)
    - [3.1 History & Motivation](#sec3_1)
    - [3.2 Encoder (Contracting Path)](#sec3_2)
    - [3.3 Bottleneck](#sec3_3)
    - [3.4 Decoder (Expanding Path)](#sec3_4)
    - [3.5 Skip Connections](#sec3_5)
    - [3.6 PyTorch vs TensorFlow/Keras — Key Differences](#sec3_6)
4. [Part 4 — State-of-the-Art Models for Brain Tumor Segmentation](#part4)
    - [4.1 nnU-Net](#sec4_1)
    - [4.2 Swin UNETR](#sec4_2)
    - [4.3 MedSAM](#sec4_3)
    - [4.4 TransBTS](#sec4_4)
    - [4.5 Attention U-Net](#sec4_5)
    - [4.6 Comparison Table](#sec4_6)
5. [Part 5 — Hands-On Implementation (PyTorch)](#part5)
    - [5.1 Setup & Imports](#sec5_1)
    - [5.2 Data Loading & Exploration (EDA)](#sec5_2)
    - [5.3 Image Visualization](#sec5_3)
    - [5.4 Dataset & DataLoader](#sec5_4)
    - [5.5 Build U-Net Model](#sec5_5)
    - [5.6 Loss Function, Optimizer & Scheduler](#sec5_6)
    - [5.7 Training & Validation Loops](#sec5_7)
    - [5.8 Run Training](#sec5_8)
    - [5.9 Training History Visualization](#sec5_9)
    - [5.10 Model Evaluation & Prediction](#sec5_10)
6. [Part 6 — Summary & Next Steps](#part6)
7. [References](#refs)

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 1 — Background & Motivation</p> <a id="part1"></a>

## <span style="color:#1a5276;font-size:22px">1.1 What is a Brain Tumor?</span> <a id="sec1_1"></a>

A **brain tumor** is an abnormal growth of cells within the brain or the central spinal canal. Brain tumors are broadly classified as:

| Category | Description | Examples |
|----------|-------------|----------|
| **Benign** | Non-cancerous, slow-growing, often have clear boundaries | Meningiomas, Schwannomas |
| **Malignant** | Cancerous, fast-growing, invade surrounding tissue | Glioblastoma (GBM), Anaplastic Astrocytoma |
| **Primary** | Originate in the brain | Gliomas, Medulloblastomas |
| **Secondary (Metastatic)** | Spread from cancer elsewhere in the body | Lung/breast cancer metastases |

**Gliomas** are the most common primary brain tumors (~30% of all brain tumors, ~80% of malignant ones). They arise from **glial cells** and are graded by the WHO from Grade I (least aggressive) to Grade IV (most aggressive — glioblastoma).

### Why Automated Segmentation Matters
- **Manual segmentation** by radiologists is time-consuming (30-60 min per scan) and subject to inter-observer variability.
- **Automated segmentation** enables faster diagnosis, treatment planning, and longitudinal monitoring.
- Essential for **surgical planning**, **radiotherapy targeting**, and **treatment response assessment**.

## <span style="color:#1a5276;font-size:22px">1.2 Medical Imaging & MRI</span> <a id="sec1_2"></a>

**Magnetic Resonance Imaging (MRI)** is the gold standard for brain tumor imaging — excellent soft-tissue contrast without ionizing radiation.

### Common MRI Sequences for Brain Tumors

| Sequence | Full Name | What It Shows |
|----------|-----------|---------------|
| **T1** | T1-weighted | Anatomy; tumors appear dark |
| **T1ce** | T1 with Contrast Enhancement (Gadolinium) | Enhancing tumor regions (active tumor) appear bright |
| **T2** | T2-weighted | Edema and tumors appear bright |
| **FLAIR** | Fluid-Attenuated Inversion Recovery | Like T2 but suppresses CSF signal; highlights perilesional edema |

In this tutorial, we work with **FLAIR abnormality** segmentation masks from the LGG (Lower-Grade Glioma) MRI dataset.

## <span style="color:#1a5276;font-size:22px">1.3 What is Image Segmentation?</span> <a id="sec1_3"></a>

**Image segmentation** assigns a class label to every pixel in an image, partitioning it into meaningful regions.

| Type | Description | Example |
|------|-------------|---------|
| **Semantic Segmentation** | Classify every pixel into a class (no instance distinction) | All tumor pixels → "tumor" class |
| **Instance Segmentation** | Distinguish individual objects of the same class | Separate tumor 1 from tumor 2 |
| **Panoptic Segmentation** | Combines semantic + instance | Full scene understanding |

For brain tumors, we use **semantic segmentation** — each pixel is **tumor** or **background** (binary).

```
Classification:  "This image contains a tumor"          → Single label
Detection:       "There is a tumor at (x, y, w, h)"     → Bounding box
Segmentation:    "These exact pixels are tumor"          → Pixel-level mask
```

## <span style="color:#1a5276;font-size:22px">1.4 Why Segmentation for Brain Tumors?</span> <a id="sec1_4"></a>

| Application | How Segmentation Helps |
|-------------|----------------------|
| **Surgical Planning** | Precise tumor boundaries guide neurosurgeons |
| **Radiation Therapy** | Accurate contours target tumor, spare healthy tissue |
| **Treatment Monitoring** | Volumetric measurements track tumor change over time |
| **Prognosis** | Tumor volume, shape, and heterogeneity correlate with outcomes |
| **Clinical Trials** | Standardized segmentation enables objective endpoints |

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 2 — Segmentation Metrics Deep Dive</p> <a id="part2"></a>

Understanding evaluation metrics is essential before building any segmentation model. In medical imaging, class imbalance is severe (tumor pixels are a small fraction of the image).

## <span style="color:#1a5276;font-size:22px">2.1 Dice Coefficient / F1 Score</span> <a id="sec2_1"></a>

The **Dice Similarity Coefficient (DSC)** measures overlap between prediction $P$ and ground truth $G$:

$$\text{Dice}(P, G) = \frac{2 \times |P \cap G|}{|P| + |G|}$$

- **Range**: 0 (no overlap) to 1 (perfect overlap)
- **Equivalent to F1 Score** at the pixel level (harmonic mean of precision and recall)
- **Dice Loss**: $\mathcal{L}_{\text{Dice}} = 1 - \text{Dice}(P, G)$

In this PyTorch tutorial, we use `segmentation_models_pytorch` (smp) which provides Dice loss via `smp.losses.DiceLoss`.

## <span style="color:#1a5276;font-size:22px">2.2 Intersection over Union (IoU / Jaccard)</span> <a id="sec2_2"></a>

$$\text{IoU}(P, G) = \frac{|P \cap G|}{|P \cup G|} = \frac{|P \cap G|}{|P| + |G| - |P \cap G|}$$

- **Range**: 0 to 1
- **Relationship to Dice**: $\text{Dice} = \frac{2 \times \text{IoU}}{1 + \text{IoU}}$
- IoU penalizes false positives and false negatives more harshly than Dice

In this tutorial, `smp.metrics.iou_score` provides both **per-image IoU** (mean IoU across images) and **dataset IoU** (aggregate intersection/union across the whole dataset).

## <span style="color:#1a5276;font-size:22px">2.3 Precision, Recall & Pixel Accuracy</span> <a id="sec2_3"></a>

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Precision** | TP / (TP + FP) | Of all pixels predicted as tumor, how many actually are? |
| **Recall (Sensitivity)** | TP / (TP + FN) | Of all actual tumor pixels, how many did we detect? |
| **Accuracy** | (TP + TN) / (TP + TN + FP + FN) | Fraction of correctly classified pixels |

**Caution**: Pixel accuracy can be misleading. If only 2% of pixels are tumor, predicting "no tumor" everywhere achieves 98% accuracy. This is why **Dice, IoU, F1, Precision, and Recall** are preferred.

In this tutorial, `smp.metrics` computes all of these from TP/FP/FN/TN statistics via `smp.metrics.get_stats()`.

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 3 — U-Net Architecture Deep Dive</p> <a id="part3"></a>

<center><img src="https://miro.medium.com/max/1200/1*f7YOaE4TWubwaFF7Z1fzNw.png" alt="U-Net Architecture" style='width:800px;'></center>

*The U-Net architecture. Left: contracting (encoder) path. Right: expanding (decoder) path. Gray arrows: skip connections. (Ronneberger et al., 2015)*

## <span style="color:#1a5276;font-size:22px">3.1 History & Motivation</span> <a id="sec3_1"></a>

**U-Net** was introduced by Ronneberger, Fischer, and Brox at MICCAI 2015, designed for **biomedical image segmentation** where:
- **Training data is scarce**
- **Precise localization is required**
- **Context and detail must coexist**

### Key Innovations
1. **Symmetric encoder-decoder** with a U-shaped structure
2. **Skip connections** that concatenate encoder features with decoder features
3. **Data augmentation** strategies (elastic deformations) for medical images

### Architecture Overview

```
Input Image (256x256x3)
    |
    v
+-----------------------------+
|     ENCODER (Contracting)    |   Captures WHAT is in the image
|  Conv -> Conv -> Pool  (x4)  |   Spatial resolution down, Channels up
+---------+-------------------+
          |
          v
+-----------------------------+
|        BOTTLENECK            |   Deepest representation
|     Conv -> Conv (1024 ch)   |
+---------+-------------------+
          |
          v
+-----------------------------+
|     DECODER (Expanding)      |   Recovers WHERE things are
|  UpConv -> Concat -> Conv(x4)|   Spatial resolution up, Channels down
+---------+-------------------+
          |
          v
    Output Mask (256x256x1)
```

## <span style="color:#1a5276;font-size:22px">3.2 Encoder (Contracting Path)</span> <a id="sec3_2"></a>

At each level:
1. **Two 3x3 convolutions** (each followed by ReLU + BatchNorm)
2. **2x2 max pooling** with stride 2 for downsampling

```
Level 1:  Input(256x256x3)  -> Conv(64) -> Conv(64)  -> Pool -> (128x128x64)
Level 2:  (128x128x64)      -> Conv(128)-> Conv(128) -> Pool -> (64x64x128)
Level 3:  (64x64x128)       -> Conv(256)-> Conv(256) -> Pool -> (32x32x256)
Level 4:  (32x32x256)       -> Conv(512)-> Conv(512) -> Pool -> (16x16x512)
```

At each deeper level, the network captures increasingly abstract features — from edges to shapes to high-level semantics.

## <span style="color:#1a5276;font-size:22px">3.3 Bottleneck</span> <a id="sec3_3"></a>

```
Bottleneck: (16x16x512) -> Conv(1024) -> Conv(1024/512) -> (16x16x512)
```

- The **bridge** between encoder and decoder
- Contains the most compressed, highest-level representation
- Has the **largest receptive field** — each "pixel" sees the entire input
- In our PyTorch implementation, the bottleneck block outputs 512 channels (for concatenation efficiency)

## <span style="color:#1a5276;font-size:22px">3.4 Decoder (Expanding Path)</span> <a id="sec3_4"></a>

At each level:
1. **2x2 transposed convolution** (`nn.ConvTranspose2d`) to upsample
2. **Concatenation** with the corresponding encoder feature map (skip connection)
3. **Two 3x3 convolutions** (ReLU + BatchNorm)

**Final layer**: A 1x1 convolution maps to 1 output channel (binary segmentation).

> **Note**: Unlike the TF/Keras version which uses sigmoid activation, this PyTorch version outputs raw **logits** (no final activation). The loss function `smp.losses.DiceLoss` with `from_logits=True` handles the sigmoid internally. For inference, we threshold at 0.5.

## <span style="color:#1a5276;font-size:22px">3.5 Skip Connections</span> <a id="sec3_5"></a>

Skip connections **concatenate** encoder feature maps to the decoder at the same spatial level, giving the decoder access to both:
- **"What"**: High-level features from the bottleneck (semantic understanding)
- **"Where"**: Fine-grained spatial features from the encoder (precise localization)

```
Encoder Level 2 output: (128x128x128)  -----+
        |                                     |  (Skip)
        v (MaxPool)                           |
  [deeper levels...]                          |
        |                                     |
        v (ConvTranspose2d to 128x128x128)    |
        +------- torch.cat(dim=1) -----------+
                        |
                        v
              (128x128x256)  <- combined channels
                   Conv -> Conv
                        v
              (128x128x128)
```

## <span style="color:#1a5276;font-size:22px">3.6 PyTorch vs TensorFlow/Keras — Key Differences</span> <a id="sec3_6"></a>

| Aspect | PyTorch (this notebook) | TensorFlow/Keras (companion notebook) |
|--------|------------------------|--------------------------------------|
| **Model definition** | Class-based (`nn.Module`) with explicit `forward()` | Functional API with `Input()` → layers → `Model()` |
| **Training loop** | Manual: forward, loss, backward, optimizer step | `model.fit()` handles everything |
| **Data loading** | `Dataset` + `DataLoader` classes | `ImageDataGenerator.flow_from_dataframe()` |
| **Concatenation** | `torch.cat([x, skip], dim=1)` (channel dim=1) | `concatenate([x, skip], axis=3)` (channel axis=3) |
| **Upsampling** | `nn.ConvTranspose2d(in, out, 2, stride=2)` | `Conv2DTranspose(out, (2,2), strides=(2,2))` |
| **Output activation** | None (raw logits) + `from_logits=True` in loss | Sigmoid in final layer |
| **Tensor format** | NCHW (batch, channels, height, width) | NHWC (batch, height, width, channels) |
| **Device management** | Explicit `.to(device)` for model and data | Automatic GPU placement |
| **Gradient control** | Manual `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` | Handled by `model.fit()` |
| **Evaluation mode** | `model.eval()` + `torch.no_grad()` | Automatic during `model.evaluate()` |

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 4 — State-of-the-Art Models for Brain Tumor Segmentation</p> <a id="part4"></a>

While U-Net remains the foundation, several advanced architectures have achieved state-of-the-art results on the **BraTS Challenge**.

## <span style="color:#1a5276;font-size:22px">4.1 nnU-Net (No-New-UNet)</span> <a id="sec4_1"></a>

A **self-configuring framework** that automatically adapts U-Net to any medical segmentation dataset. It fingerprints the dataset and auto-selects preprocessing, architecture (2D, 3D full-res, or 3D cascade U-Net), training scheme, and post-processing.

```
Input Dataset -> Dataset Fingerprinting -> Pipeline Selection -> Auto Preprocessing
     -> Auto Training (Dice+CE loss) -> Auto Post-processing -> Optimized Segmentation
```

- **Winner/top performer** in 33+ medical segmentation challenges
- No manual tuning required
- **Reference**: Isensee et al., *Nature Methods* 2021. [DOI:10.1038/s41592-020-01008-z](https://doi.org/10.1038/s41592-020-01008-z) | [GitHub](https://github.com/MIC-DKFZ/nnUNet)

## <span style="color:#1a5276;font-size:22px">4.2 Swin UNETR</span> <a id="sec4_2"></a>

Replaces the CNN encoder with a **Swin Transformer** (shifted-window self-attention) while keeping the U-Net decoder. Won **BraTS 2021**.

```
Input 3D MRI -> Patch Partition -> Swin Transformer Stages (x4, with skip connections)
     -> Bottleneck -> CNN Decoder (Deconv + Concat + Conv) -> Multi-class Segmentation
```

- **Shifted-window attention**: Global context with linear complexity
- **Self-supervised pre-training** on 5,050 unlabeled CT scans
- **BraTS 2021**: Dice 91.6% (WT), 86.5% (TC), 83.1% (ET)
- **Reference**: Hatamizadeh et al., *BrainLes/MICCAI 2021*. [arXiv:2201.01266](https://arxiv.org/abs/2201.01266) | Tang et al., *CVPR 2022*. [arXiv:2111.14791](https://arxiv.org/abs/2111.14791) | [GitHub](https://github.com/Project-MONAI/tutorials/blob/main/3d_segmentation/swin_unetr_brats21_segmentation_3d.ipynb)

## <span style="color:#1a5276;font-size:22px">4.3 MedSAM (Medical Segment Anything Model)</span> <a id="sec4_3"></a>

Adapts Meta's **Segment Anything Model (SAM)** to medical imaging. Fine-tuned on 1.5M+ medical image-mask pairs across 10+ modalities.

```
Input Image -> ViT Image Encoder -> Image Embeddings
Bounding Box/Points -> Prompt Encoder -> Prompt Embeddings
     -> Cross-Attention Mask Decoder -> Segmentation Mask
```

- **Foundation model**: One model across multiple organs, modalities, and tasks
- **Interactive**: Users provide bounding boxes or point prompts
- **Limitation**: Prompt-dependent, 2D only
- **Reference**: Ma et al., *Nature Communications* 2024. [DOI:10.1038/s41467-024-44824-z](https://doi.org/10.1038/s41467-024-44824-z) | [GitHub](https://github.com/bowang-lab/MedSAM) | [Brain](https://github.com/vpulab/med-sam-brain)

## <span style="color:#1a5276;font-size:22px">4.4 TransBTS</span> <a id="sec4_4"></a>

First architecture to introduce **Transformers into brain tumor segmentation**. Uses 3D CNN encoder for local features, Transformer at the bottleneck for global context, then CNN decoder.

```
Input 3D MRI -> 3D CNN Encoder (x4 stages, with skips) -> Reshape to sequence
     -> Transformer (Positional Encoding + Multi-Head Self-Attention)
     -> Reshape to 3D -> CNN Decoder (Upsample + Concat + Conv) -> Segmentation
```

- **Hybrid CNN-Transformer**: Local features + global dependencies
- **BraTS 2019**: Dice 88.4% (WT), 83.6% (TC), 78.9% (ET)
- **Reference**: Wang et al., *MICCAI 2021*. [DOI:10.1007/978-3-030-87193-2_11](https://doi.org/10.1007/978-3-030-87193-2_11) | [GitHub](https://github.com/Wenxuan-1119/TransBTS)

## <span style="color:#1a5276;font-size:22px">4.5 Attention U-Net</span> <a id="sec4_5"></a>

Enhances U-Net with **Attention Gates (AGs)** at skip connections. Instead of blindly concatenating, attention gates learn to suppress irrelevant regions and highlight salient features.

```
Encoder features (skip): Fi    Decoder features (gating): gi
         |                              |
         v                              v
     [Attention Gate: Wg*gi + Wx*Fi -> ReLU -> psi -> sigmoid -> alpha]
         |
   Output: alpha * Fi  (reweighted skip)  ->  Concat with Decoder -> Conv
```

- **No additional supervision needed** — attention learned from segmentation loss
- **Improved small-object segmentation**
- **Reference**: Oktay et al., *MIDL 2018*. [arXiv:1804.03999](https://arxiv.org/abs/1804.03999)

## <span style="color:#1a5276;font-size:22px">4.6 Model Comparison</span> <a id="sec4_6"></a>

| Model | Year | Venue | Key Innovation | BraTS Dice (WT) | Best For |
|-------|------|-------|----------------|-----------------|----------|
| **U-Net** | 2015 | MICCAI | Skip connections | ~87% | Quick baseline, limited compute |
| **nnU-Net** | 2021 | Nat. Methods | Self-configuring | ~91% | Best out-of-the-box |
| **Swin UNETR** | 2021 | BrainLes/CVPR | Shifted-window attention | ~92% | SOTA with large GPU budget |
| **MedSAM** | 2024 | Nat. Commun. | Foundation model | ~89%* | Interactive/clinical workflow |
| **TransBTS** | 2021 | MICCAI | Hybrid CNN-Transformer | ~88% | 3D volumetric research |
| **Attention U-Net** | 2018 | MIDL | Attention gates | ~88% | Drop-in U-Net improvement |

*\*MedSAM Dice varies based on prompt quality.*

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 5 — Hands-On Implementation (PyTorch)</p> <a id="part5"></a>

Now let's build a complete brain tumor segmentation pipeline using **U-Net in PyTorch**.

**Dataset**: LGG MRI Segmentation — 3,929 brain MRI images from 110 patients with lower-grade gliomas.

**Key libraries**:
- `torch` / `torchvision` — model building and training
- `segmentation_models_pytorch (smp)` — Dice loss and evaluation metrics
- `torchmetrics` — additional metrics
- `OpenCV` / `scikit-image` — image processing

## <span style="color:#1a5276;font-size:22px">5.1 Setup & Imports</span> <a id="sec5_1"></a>

In [ ]:
!python --version

In [ ]:
!pip install segmentation-models-pytorch torch torchvision
!pip install -qqq torchmetrics
!pip install -U git+https://github.com/qubvel/segmentation_models.pytorch

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import glob

import gc
import time

from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchmetrics

from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
from IPython.display import Image
from skimage import io

import segmentation_models_pytorch as smp
from segmentation_models_pytorch.datasets import SimpleOxfordPetDataset

from pprint import pprint

from sklearn.model_selection import train_test_split
import cv2
from sklearn.preprocessing import StandardScaler, normalize
from IPython.display import display

from PIL import Image

import torchvision
from torchvision import transforms

In [ ]:
start_time = datetime.now()

## <span style="color:#1a5276;font-size:22px">5.2 Data Loading & Exploration (EDA)</span> <a id="sec5_2"></a>

The dataset is organized as patient directories, each containing MRI images and corresponding masks. We build a DataFrame mapping each image to its mask, then add a binary `mask` column indicating whether the image contains a tumor.

### Key steps:
1. Glob all files from the dataset directory
2. Separate images from masks (masks contain "mask" in filename)
3. Sort and pair them
4. Create a diagnosis column by checking if the mask has non-zero pixels

In [ ]:
# Auto-detect dataset path
import os
candidate_paths = [
    '/kaggle/input/lgg-mri-segmentation/kaggle_3m',
    '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m',
    '/kaggle/input/lgg-mri-segmentation',
    '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation',
]
data_dir = None
for p in candidate_paths:
    if os.path.isdir(p):
        data_dir = p
        break
if data_dir is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if any('_mask' in f for f in files):
            data_dir = os.path.dirname(root)
            break
assert data_dir is not None, "Dataset not found! Please check the dataset path."
print(f"Using dataset path: {data_dir}")

img_data = pd.read_csv(os.path.join(data_dir, 'data.csv'))
img_data.head()

In [ ]:
img_data.info()

In [ ]:
img_data.shape

In [ ]:
data_path = []
for sub_dir_path in glob.glob(data_dir + "/*"):
    try:
        dir_name = sub_dir_path.split('/')[-1]
        for filename in os.listdir(sub_dir_path):
            mask_path = sub_dir_path + '/' + filename
            data_path.extend([dir_name, mask_path])
    except Exception as e:
        print(e)

In [ ]:
filenames = data_path[::2]
masks = data_path[1::2]

In [ ]:
df = pd.DataFrame(data={"patient_id": filenames,"img_path": masks})
print(df.shape)
df

### Separate images and masks

We filter the DataFrame to separate original images from mask images using string matching on the file path.

In [ ]:
original_img = df[~df['img_path'].str.contains("mask")]
mask_img = df[df['img_path'].str.contains("mask")]

In [ ]:
original_img, mask_img

In [ ]:
imgs = sorted(original_img["img_path"].values, key=lambda x : int(x.split('/')[-1].split('.')[0].split('_')[-1]) if x.split('/')[-1].split('.')[0].split('_')[-1].isdigit() else 0)
masks = sorted(mask_img["img_path"].values, key=lambda x : int(x.split('/')[-1].replace('_mask','').split('.')[0].split('_')[-1]) if x.split('/')[-1].replace('_mask','').split('.')[0].split('_')[-1].isdigit() else 0)

# Sorting check
idx = random.randint(0, len(imgs)-1)
print("Image path:", imgs[idx], "\nMask path:", masks[idx])

In [ ]:
mri_df = pd.DataFrame({"patient_id": original_img.patient_id.values,"img_path": imgs,
                           'mask_path':masks})
mri_df

### Add diagnosis label

We create a function that reads each mask and checks if it has any non-zero pixels (indicating tumor presence).

In [ ]:
def get_diagnosis(img_path):
    value = np.max(cv2.imread(img_path))
    if value > 0 : 
        return 1
    else:
        return 0

In [ ]:
mri_df['mask'] = mri_df['mask_path'].apply(lambda x: get_diagnosis(x))

mri_df['mask_path'] = mri_df['mask_path'].apply(lambda x: str(x))

print(mri_df.shape)
mri_df

In [ ]:
mri_df.drop(columns=['patient_id'],inplace=True)

### Check class balance

Understanding the distribution of tumor vs no-tumor images is important. Class imbalance affects training — if most images have no tumor, the model may learn to predict "no tumor" by default.

In [ ]:
mri_df['mask'].value_counts().plot(kind='bar',color=['g','r'],
                title='Count of Tumour vs No Tumour')

In [ ]:
mri_df['mask'].value_counts()

## <span style="color:#1a5276;font-size:22px">5.3 Image Visualization</span> <a id="sec5_3"></a>

Let's visualize brain MRI images alongside their tumor masks. The third column shows the MRI with the mask overlaid in red — this is what the model needs to learn to predict.

In [ ]:
count = 0
i = 0
fig,axs = plt.subplots(3,3, figsize=(20,15))
for mask in mri_df['mask']:
    if (mask==1):
        img = io.imread(mri_df.img_path[i])
        print(img.shape)
        axs[count][0].title.set_text("Brain MRI")
        axs[count][0].imshow(img)
        
        mask = io.imread(mri_df.mask_path[i])
        axs[count][1].title.set_text("Mask =" + str(mri_df['mask'][i]))
        axs[count][1].imshow(mask, cmap='gray')
        
        img[mask==255] = (255,0,0)  # change pixel color at the position of mask
        axs[count][2].title.set_text("MRI with Mask =" + str(mri_df['mask'][i]))
        axs[count][2].imshow(img)
        count +=1
    i += 1
    if (count==3):
        break
        
fig.tight_layout()

## <span style="color:#1a5276;font-size:22px">5.4 Dataset & DataLoader</span> <a id="sec5_4"></a>

In PyTorch, we create a custom `Dataset` class and use `DataLoader` for batching:

### PyTorch Data Pipeline

```
Dataset (defines how to load ONE sample)
    |
    v
DataLoader (batches + shuffles + multiprocessing)
    |
    v
Training loop (iterates over batches)
```

**Key components**:
- **`adjust_data()`**: Normalizes images to [0,1] and binarizes masks
- **`MyDataset`**: Custom `Dataset` that loads image-mask pairs, applies transforms
- **`prepare_loaders()`**: Splits data and creates train/valid/test DataLoaders

In [ ]:
image_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

mask_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    ])

In [ ]:
def adjust_data(img, mask):
    img = img / 255.
    mask = mask / 255.
    mask[mask > 0.5] = 1.0
    mask[mask <= 0.5] = 0.0
    
    return (img, mask)

In [ ]:
class MyDataset(Dataset):
    def __init__(self, df= mri_df, 
                 adjust_data = adjust_data, 
                 image_transform=image_transform, mask_transform=mask_transform):
        self.df = df
        self.image_transform = image_transform
        self.mask_transform = mask_transform
        self.adjust_data= adjust_data

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.df.loc[idx, 'img_path']
        mask_path = self.df.loc[idx, 'mask_path']

        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path)
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        image, mask = self.adjust_data(image, mask)

        if self.image_transform:
            image = self.image_transform(image).float()

        if self.mask_transform:
            mask = self.mask_transform(mask)
        return image, mask

### Verify a sample

In [ ]:
index = 2911
data = MyDataset()[index]
data[0].shape, data[1].shape

In [ ]:
plt.imshow(data[0].permute(1, 2, 0).numpy())

In [ ]:
plt.imshow(data[1].permute(1, 2, 0).squeeze(-1).numpy())

In [ ]:
np.unique(data[1])

### Create DataLoaders

We split the data into 65% train, 20% validation, 15% test.

In [ ]:
def prepare_loaders(df= mri_df,
                    train_num= int(mri_df.shape[0] * .6), 
                    valid_num= int(mri_df.shape[0] * .8), 
                    bs = 32):
    
    train = df[:train_num].reset_index(drop=True)
    valid = df[train_num : valid_num].reset_index(drop=True)    
    test  = df[valid_num:].reset_index(drop=True)

    train_ds = MyDataset(df = train)
    valid_ds = MyDataset(df = valid)
    test_ds = MyDataset(df = test)

    train_loader = DataLoader(train_ds, batch_size = bs, num_workers = os.cpu_count(), shuffle = True)
    valid_loader = DataLoader(valid_ds, batch_size = bs, num_workers = os.cpu_count(), shuffle = False)
    test_loader = DataLoader(test_ds, batch_size = 4, num_workers = os.cpu_count(), shuffle = True)
    
    print("DataLoader Completed")
    
    return train_loader, valid_loader, test_loader

In [ ]:
train_loader, valid_loader, test_loader = prepare_loaders(df= mri_df,
                                                            train_num= int(mri_df.shape[0] * .65), 
                                                            valid_num= int(mri_df.shape[0] * .85), 
                                                            bs = 32)

In [ ]:
data = next(iter(train_loader))
data[0].shape, data[1].shape

## <span style="color:#1a5276;font-size:22px">5.5 Build U-Net Model</span> <a id="sec5_5"></a>

Our PyTorch U-Net is built from two classes:

1. **`Block`** — A reusable encoder/decoder block: `Conv2d -> ReLU -> Conv2d -> BatchNorm -> ReLU -> MaxPool`
   - Returns **two** outputs: the pooled result (for the next level) and the pre-pool features (for skip connections)

2. **`UNet`** — The full architecture:
   - 4 encoder blocks + 1 bottleneck block
   - 4 decoder blocks with `ConvTranspose2d` upsampling and `torch.cat` skip connections
   - Final 1x1 convolution to produce 1-channel output (logits, no activation)

In [ ]:
class Block(nn.Module):
    def __init__(self, inputs = 3, middles = 64, outs = 64):
        super().__init__()
        
        self.conv1 = nn.Conv2d(inputs, middles, 3, 1, 1)
        self.conv2 = nn.Conv2d(middles, outs, 3, 1, 1)
        self.relu = nn.ReLU()
        self.bn = nn.BatchNorm2d(outs)
        self.pool = nn.MaxPool2d(2, 2)
        
    def forward(self, x):
        
        x = self.relu(self.conv1(x))
        x = self.relu(self.bn(self.conv2(x)))
        
        return self.pool(x), x
        # self.pool(x): [bs, out, h*.5, w*.5]  -> passed to next encoder level
        # x: [bs, out, h, w]                    -> skip connection

In [ ]:
class UNet(nn.Module):
    def __init__(self,):
        super().__init__()
        
        self.en1 = Block(3, 64, 64)
        self.en2 = Block(64, 128, 128)
        self.en3 = Block(128, 256, 256)
        self.en4 = Block(256, 512, 512)
        self.en5 = Block(512, 1024, 512)
        
        self.upsample4 = nn.ConvTranspose2d(512, 512, 2, stride = 2)
        self.de4 = Block(1024, 512, 256)
        
        self.upsample3 = nn.ConvTranspose2d(256, 256, 2, stride = 2)
        self.de3 = Block(512, 256, 128)
        
        self.upsample2 = nn.ConvTranspose2d(128, 128, 2, stride = 2)
        self.de2 = Block(256, 128, 64)
        
        self.upsample1 = nn.ConvTranspose2d(64, 64, 2, stride = 2)
        self.de1 = Block(128, 64, 64)
        
        self.conv_last = nn.Conv2d(64, 1, kernel_size=1, stride = 1, padding = 0)
        
    def forward(self, x):
        # x: [bs, 3, 256, 256]
        
        x, e1 = self.en1(x)
        # x: [bs, 64, 128, 128]
        # e1: [bs, 64, 256, 256]
        
        x, e2 = self.en2(x)
        # x: [bs, 128, 64, 64]
        # e2: [bs, 128, 128, 128]
        
        x, e3 = self.en3(x)
        # x: [bs, 256, 32, 32]
        # e3: [bs, 256, 64, 64]
        
        x, e4 = self.en4(x)
        # x: [bs, 512, 16, 16]
        # e4: [bs, 512, 32, 32]
        
        _, x = self.en5(x)
        # x: [bs, 512, 16, 16]
        
        x = self.upsample4(x)
        # x: [bs, 512, 32, 32]
        x = torch.cat([x, e4], dim=1)
        # x: [bs, 1024, 32, 32]
        _,  x = self.de4(x)
        # x: [bs, 256, 32, 32]
        
        x = self.upsample3(x)
        # x: [bs, 256, 64, 64]
        x = torch.cat([x, e3], dim=1)
        # x: [bs, 512, 64, 64]
        _, x = self.de3(x)
        # x: [bs, 128, 64, 64]
        
        x = self.upsample2(x)
        # x: [bs, 128, 128, 128]
        x = torch.cat([x, e2], dim=1)
        # x: [bs, 256, 128, 128]
        _, x = self.de2(x)
        # x: [bs, 64, 128, 128]
        
        x = self.upsample1(x)
        # x: [bs, 64, 256, 256]
        x = torch.cat([x, e1], dim=1)
        # x: [bs, 128, 256, 256]
        _, x = self.de1(x)
        # x: [bs, 64, 256, 256]
        
        x = self.conv_last(x)
        # x: [bs, 1, 256, 256]
        
        return x

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
device

In [ ]:
model = UNet().to(device)
model

## <span style="color:#1a5276;font-size:22px">5.6 Loss Function, Optimizer & Scheduler</span> <a id="sec5_6"></a>

| Component | Choice | Rationale |
|-----------|--------|-----------|
| **Loss** | `smp.losses.DiceLoss(BINARY_MODE, from_logits=True)` | Directly optimizes Dice; handles logits internally |
| **Optimizer** | Adam (default lr=0.001) | Adaptive learning rate, good default for segmentation |
| **Scheduler** | CosineAnnealingLR (T_max=200, eta_min=1e-6) | Smoothly decays learning rate following cosine curve |

> **Why Dice Loss over BCE?** BCE treats each pixel independently and is dominated by the background class. Dice loss directly optimizes the overlap metric we care about, handling class imbalance inherently.

In [ ]:
loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
optimizer = torch.optim.Adam(model.parameters(), )

In [ ]:
from torch.optim import lr_scheduler

scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max = 200, eta_min = 1e-6)

## <span style="color:#1a5276;font-size:22px">5.7 Training & Validation Loops</span> <a id="sec5_7"></a>

In PyTorch, we write explicit training and validation loops. This gives full control over the training process.

### Training loop (per epoch):
```
model.train()                  # Enable training mode (BatchNorm, Dropout active)
for batch in dataloader:
    x, y_true = batch          # Load batch
    y_pred = model(x)          # Forward pass
    loss = loss_fn(y_pred, y)  # Compute loss
    optimizer.zero_grad()      # Clear old gradients
    loss.backward()            # Backpropagation
    optimizer.step()           # Update weights
    scheduler.step()           # Update learning rate
```

### Validation loop (per epoch):
```
model.eval()                   # Disable BatchNorm/Dropout
with torch.no_grad():          # No gradient computation (saves memory)
    for batch in dataloader:
        y_pred = model(x)      # Forward pass only
        loss = loss_fn(y_pred, y)
```

### Metrics tracked per epoch:
Loss, Accuracy, Precision, Recall, Dice/F1 Score, Per-Image IoU, Dataset IoU

In [ ]:
def train_one_epoch(model = model, 
                    dataloader = train_loader, 
                    loss_fn = loss_fn, 
                    optimizer = optimizer,
                    scheduler = None,
                    device = device, 
                    epoch = 1):
    model.train() 
    train_loss, dataset_size = 0,  0
    
    bar = tqdm(dataloader, total = len(dataloader))
    tp_l, fp_l, fn_l, tn_l = [], [], [], []
    
    for data in bar:
        x = data[0].to(device)     
        y_true = data[1].to(device) 
        y_pred = model(x)          
        
        loss = loss_fn(y_pred, y_true)
        
        pred_mask = (y_pred > 0.5).float()
        btp, bfp, bfn, btn = smp.metrics.get_stats(pred_mask.long(), y_true.long(), mode="binary")

        # 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if scheduler is not None:
            scheduler.step()
        
        # train_epoch_loss
        bs = x.shape[0]
        dataset_size += bs
        train_loss += (loss.item() * bs)
        train_epoch_loss = train_loss / dataset_size
        
        tp_l.append(btp)
        fp_l.append(bfp)
        fn_l.append(bfn)
        tn_l.append(btn)
        
        tp = torch.cat(tp_l)
        fp = torch.cat(fp_l)
        fn = torch.cat(fn_l)
        tn = torch.cat(tn_l)
        
        recall = smp.metrics.recall(tp, fp, fn, tn, reduction="micro")
        precision = smp.metrics.precision(tp, fp, fn, tn, reduction="micro")
        
        f1_score = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")
        dice_score = f1_score  # Dice coefficient = F1 score for binary segmentation
        accuracy = smp.metrics.accuracy(tp, fp, fn, tn, reduction="macro")
        
        per_image_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro-imagewise")
        dataset_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")

        bar.set_description(f"EP:{epoch} | TL:{train_epoch_loss:.3e} | ACC: {accuracy:.2f} | Dice/F1: {dice_score:.3f} ")
        
    metrics =  dict()
    
    metrics['f1_score'] = f1_score.detach().cpu().item()
    metrics['dice_score'] = dice_score.detach().cpu().item()  # same as f1 for binary
    metrics['accuracy'] = accuracy.detach().cpu().item()
    
    metrics['recall'] = recall.detach().cpu().item()
    metrics['precision'] = precision.detach().cpu().item()
    
    metrics['dataset_iou'] = dataset_iou.detach().cpu().item()
    metrics['per_iou'] = per_image_iou.detach().cpu().item()
    
    metrics['loss'] = train_epoch_loss

    return metrics

In [ ]:
@torch.no_grad()
def valid_one_epoch(model = model, 
                    dataloader = valid_loader, 
                    loss_fn = loss_fn,
                    device = device, 
                    epoch = 0):
    model.eval() 
    valid_loss, dataset_size = 0,  0
    bar = tqdm(dataloader, total = len(dataloader))
    tp_l, fp_l, fn_l, tn_l = [], [], [], []
    
    with torch.no_grad():
        for data in bar:
            x = data[0].to(device)     
            y_true = data[1].to(device) 
            y_pred = model(x)        
            
            loss = loss_fn(y_pred, y_true)
            
            pred_mask = (y_pred > 0.5).float()
            btp, bfp, bfn, btn = smp.metrics.get_stats(pred_mask.long(), y_true.long(), mode="binary")

            tp_l.append(btp)
            fp_l.append(bfp)
            fn_l.append(bfn)
            tn_l.append(btn)

            tp = torch.cat(tp_l)
            fp = torch.cat(fp_l)
            fn = torch.cat(fn_l)
            tn = torch.cat(tn_l)

            recall = smp.metrics.recall(tp, fp, fn, tn, reduction="micro")
            precision = smp.metrics.precision(tp, fp, fn, tn, reduction="micro")

            f1_score = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")
            dice_score = f1_score  # Dice coefficient = F1 score for binary segmentation
            accuracy = smp.metrics.accuracy(tp, fp, fn, tn, reduction="macro")

            per_image_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro-imagewise")
            dataset_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")

            bs = x.shape[0]
            dataset_size += bs
            valid_loss += (loss.item() * bs)
            valid_epoch_loss = valid_loss / dataset_size

            bar.set_description(f"EP:{epoch} | VL:{valid_epoch_loss:.3e} | ACC: {accuracy:.2f} | Dice/F1: {dice_score:.3f} ")

    metrics =  dict()
    
    metrics['f1_score'] = f1_score.detach().cpu().item()
    metrics['dice_score'] = dice_score.detach().cpu().item()
    metrics['accuracy'] = accuracy.detach().cpu().item()
    
    metrics['recall'] = recall.detach().cpu().item()
    metrics['precision'] = precision.detach().cpu().item()
    
    metrics['dataset_iou'] = dataset_iou.detach().cpu().item()
    metrics['per_iou'] = per_image_iou.detach().cpu().item()
    
    metrics['loss'] = valid_epoch_loss

    return metrics

## <span style="color:#1a5276;font-size:22px">5.8 Run Training</span> <a id="sec5_8"></a>

The `run_training` function orchestrates the full training process:
- Runs `train_one_epoch` and `valid_one_epoch` for each epoch
- Tracks all metrics history
- Saves the best model based on **Dice/F1 score** and **lowest loss** (two separate checkpoints)
- Implements **early stopping** if validation loss doesn't improve for 20 epochs

In [ ]:
import copy

def run_training(model = model, 
                 loss_fn = loss_fn, 
                 train_loader = train_loader,
                 valid_loader = valid_loader,
                 optimizer = optimizer, 
                 device = device, 
                 n_epochs=100, 
                 early_stop = 20,
                 scheduler = None):

    if torch.cuda.is_available():
        print("INFO: GPU - {}\n".format(torch.cuda.get_device_name()))
    
    start = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())

    lowest_epoch, lowest_loss = np.inf, np.inf
    
    train_history, valid_history = [],  []
    train_recalls, valid_recalls = [],  []
    
    train_pres, valid_pres = [],  []
    train_accs, valid_accs = [],  []
    
    train_f1s, valid_f1s = [],  []
    train_dices, valid_dices = [], []
    
    train_per_ious, valid_per_ious = [], []
    train_dataset_ious, valid_dataset_ious = [], []
    
    print_iter = 5

    best_score = 0
    best_model = "None"

    for epoch in range(0, n_epochs):
        gc.collect()

        train_metrics = train_one_epoch(model= model,
                                       dataloader = train_loader,
                                       optimizer = optimizer,
                                       scheduler = scheduler,
                                       device = device,
                                       epoch = epoch + 1
                                       )
        
        valid_metrics = valid_one_epoch(model,
                                       dataloader = valid_loader,
                                       device = device,
                                       epoch = epoch + 1)
        
        train_history += [train_metrics['loss']]
        valid_history += [valid_metrics['loss']]
        
        train_recalls += [train_metrics['recall']]
        valid_recalls += [valid_metrics['recall']]
        
        train_pres += [train_metrics['precision']]
        valid_pres += [valid_metrics['precision']]
        
        train_accs += [train_metrics['accuracy']]
        valid_accs += [valid_metrics['accuracy']]
        
        train_f1s += [train_metrics['f1_score']]
        valid_f1s += [valid_metrics['f1_score']]
        
        train_dices += [train_metrics['dice_score']]
        valid_dices += [valid_metrics['dice_score']]
        
        train_per_ious += [train_metrics['per_iou']]
        valid_per_ious += [valid_metrics['per_iou']]
        
        train_dataset_ious += [train_metrics['dataset_iou']]
        valid_dataset_ious += [valid_metrics['dataset_iou']]
        
        
        print()
        if (epoch + 1) % print_iter == 0:
            print(f"Epoch:{epoch + 1}|TL:{train_metrics['loss']:.3e}|VL:{valid_metrics['loss']:.3e}|Dice:{valid_metrics['dice_score']:.4f}|F1:{valid_metrics['f1_score']:.4f}|Dataset IOU:{valid_metrics['dataset_iou']:.4f}|Per Img IOU:{valid_metrics['per_iou']:.4f}|")
            print()
            
        if best_score < valid_metrics['f1_score']:
            print(f"Validation F1 Improved({best_score:.2f}) --> ({ valid_metrics['f1_score']:.2f})")
            best_model = model
            best_score = valid_metrics['f1_score']
            best_model = copy.deepcopy(model.state_dict())
            PATH2 =  f"model_f1.bin"
            torch.save(model.state_dict(), PATH2)
            print(f"Better_F1_Model Saved")
            print()

        if valid_metrics['loss']< lowest_loss:
            print(f"Validation Loss Improved({lowest_loss:.4e}) --> ({ valid_metrics['loss']:.4e})")
            lowest_loss = valid_metrics['loss']
            lowest_epoch = epoch
            best_model_wts = copy.deepcopy(model.state_dict())
            PATH = f"model.bin"
            torch.save(model.state_dict(), PATH)
            print(f"Better Loss Model Saved")
            print()
        else:
            if early_stop > 0 and lowest_epoch + early_stop < epoch + 1:
                print("Stopping... no improvement!")
                break
                
    print()
    end = time.time()
    time_elapsed = end - start
    print('Training complete in {:.0f}h {:.0f}m {:.0f}s'.format(time_elapsed // 3600, (time_elapsed % 3600) // 60, (time_elapsed % 3600) % 60))
    print("Best Loss: %.4e at %d th Epoch" % (lowest_loss, lowest_epoch))

    model.load_state_dict(torch.load('./model_f1.bin'))

    result = dict()
    result["Train Loss"] = train_history
    result["Valid Loss"] = valid_history
    
    result["Train Recall"] = train_recalls
    result["Valid Recall"] = valid_recalls
    
    result["Train Precision"] = train_pres
    result["Valid Precision"] = valid_pres
    
    result["Train Accuracy"] = train_accs
    result["Valid Accuracy"] = valid_accs
    
    result["Train F1 Score"] = train_f1s
    result["Valid F1 Score"] = valid_f1s
    
    result["Train Dice"] = train_dices
    result["Valid Dice"] = valid_dices
    
    result["Train per Image IOU"] = train_per_ious
    result["Valid per Image IOU"] = valid_per_ious
    
    result["Train Dataset IOU"] = train_dataset_ious
    result["Valid Dataset IOU"] = valid_dataset_ious
    
    return model, result

In [ ]:
model, result = run_training(model = model, 
                             loss_fn = loss_fn, 
                             optimizer = optimizer, 
                             device = device, 
                             scheduler = scheduler,
                             n_epochs = 50)

## <span style="color:#1a5276;font-size:22px">5.9 Training History Visualization</span> <a id="sec5_9"></a>

We visualize 7 metrics to diagnose training behavior: Loss, Accuracy, Recall, Precision, **Dice Coefficient** (= F1 Score), Per-Image IoU, and Dataset IoU.

In [ ]:
## Train/Valid Loss History
plot_from = 0
plt.figure(figsize=(20, 10))
plt.title("Train/Valid Loss History", fontsize = 20)
plt.plot(
    range(0, len(result['Train Loss'][plot_from:])), 
    result['Train Loss'][plot_from:], 
    label = 'Train Loss'
    )

plt.plot(
    range(0, len(result['Valid Loss'][plot_from:])), 
    result['Valid Loss'][plot_from:], 
    label = 'Valid Loss'
    )

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
## Train/Valid Accuracy History
plot_from = 0
plt.figure(figsize=(20, 10))
plt.title("Train/Valid Accuracy History", fontsize = 20)
plt.plot(range(0, len(result['Train Accuracy'][plot_from:])), result['Train Accuracy'][plot_from:], label = 'Train Accuracy')
plt.plot(range(0, len(result['Valid Accuracy'][plot_from:])), result['Valid Accuracy'][plot_from:], label = 'Valid Accuracy')
plt.legend()
plt.grid(True)

In [ ]:
## Train/Valid Recall History
plot_from = 0
plt.figure(figsize=(20, 10))
plt.title("Train/Valid Recall History", fontsize = 20)
plt.plot(range(0, len(result['Train Recall'][plot_from:])), result['Train Recall'][plot_from:], label = 'Train Recall')
plt.plot(range(0, len(result['Valid Recall'][plot_from:])), result['Valid Recall'][plot_from:], label = 'Valid Recall')
plt.legend()
plt.grid(True)

In [ ]:
## Train/Valid Precision History
plot_from = 0
plt.figure(figsize=(20, 10))
plt.title("Train/Valid Precision History", fontsize = 20)
plt.plot(range(0, len(result['Train Precision'][plot_from:])), result['Train Precision'][plot_from:], label = 'Train Precision')
plt.plot(range(0, len(result['Valid Precision'][plot_from:])), result['Valid Precision'][plot_from:], label = 'Valid Precision')
plt.legend()
plt.grid(True)

In [ ]:
## Train/Valid Dice / F1 Score History
plot_from = 0
plt.figure(figsize=(20, 10))
plt.title("Train/Valid Dice Coefficient (= F1 Score) History", fontsize = 20)
plt.plot(range(0, len(result['Train Dice'][plot_from:])), result['Train Dice'][plot_from:], label = 'Train Dice')
plt.plot(range(0, len(result['Valid Dice'][plot_from:])), result['Valid Dice'][plot_from:], label = 'Valid Dice')
plt.legend()
plt.grid(True)

In [ ]:
## Train/Valid Per Image IOU History
plot_from = 0
plt.figure(figsize=(20, 10))
plt.title("Train/Valid per Image IOU History", fontsize = 20)
plt.plot(range(0, len(result['Train per Image IOU'][plot_from:])), result['Train per Image IOU'][plot_from:], label = 'Train per Image IOU')
plt.plot(range(0, len(result['Valid per Image IOU'][plot_from:])), result['Valid per Image IOU'][plot_from:], label = 'Valid per Image IOU')
plt.legend()
plt.grid(True)

In [ ]:
## Train/Valid Dataset IOU History
plot_from = 0
plt.figure(figsize=(20, 10))
plt.title("Train/Valid Dataset IOU History", fontsize = 20)
plt.plot(range(0, len(result['Train Dataset IOU'][plot_from:])), result['Train Dataset IOU'][plot_from:], label = 'Train Dataset IOU')
plt.plot(range(0, len(result['Valid Dataset IOU'][plot_from:])), result['Valid Dataset IOU'][plot_from:], label = 'Valid Dataset IOU')
plt.legend()
plt.grid(True)

## <span style="color:#1a5276;font-size:22px">5.10 Model Evaluation & Prediction</span> <a id="sec5_10"></a>

We load the best model (by Dice/F1 score) and visualize predictions on test batches. For each image we show:
1. **Original MRI** — the input
2. **Ground truth mask** — manual annotation
3. **Predicted mask** — model output thresholded at 0.5

In [ ]:
# Load best F1 model
model.load_state_dict(torch.load('/kaggle/working/model_f1.bin'))

In [ ]:
batch = next(iter(test_loader))
with torch.no_grad():
    model.eval()
    logits = model(batch[0].to(device))
pr_masks = (logits.squeeze(1) > 0.5).float()

for image, gt_mask, pr_mask in zip(batch[0], batch[1], pr_masks):
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(image.numpy().transpose(1, 2, 0))  # convert CHW -> HWC
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask.numpy().squeeze(), cmap = 'gray')
    plt.title("Ground truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pr_mask.detach().cpu().numpy())
    plt.title("Prediction")
    plt.axis("off")

    plt.show()

In [ ]:
batch = next(iter(test_loader))
with torch.no_grad():
    model.eval()
    logits = model(batch[0].to(device))
pr_masks = (logits.squeeze(1) > 0.5).float()

for image, gt_mask, pr_mask in zip(batch[0], batch[1], pr_masks):
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(image.numpy().transpose(1, 2, 0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask.numpy().squeeze(), cmap = 'gray')
    plt.title("Ground truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pr_mask.detach().cpu().numpy())
    plt.title("Prediction")
    plt.axis("off")

    plt.show()

In [ ]:
batch = next(iter(test_loader))
with torch.no_grad():
    model.eval()
    logits = model(batch[0].to(device))
pr_masks = (logits.squeeze(1) > 0.5).float()

for image, gt_mask, pr_mask in zip(batch[0], batch[1], pr_masks):
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(image.numpy().transpose(1, 2, 0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask.numpy().squeeze(), cmap = 'gray')
    plt.title("Ground truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pr_mask.detach().cpu().numpy())
    plt.title("Prediction")
    plt.axis("off")

    plt.show()

### Better Loss Model

Let's also check predictions from the model saved based on lowest validation loss (may differ from the F1-best model).

In [ ]:
model.load_state_dict(torch.load('/kaggle/working/model.bin'))

In [ ]:
batch = next(iter(test_loader))
with torch.no_grad():
    model.eval()
    logits = model(batch[0].to(device))
pr_masks = (logits.squeeze(1) > 0.5).float()

for image, gt_mask, pr_mask in zip(batch[0], batch[1], pr_masks):
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(image.numpy().transpose(1, 2, 0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt_mask.numpy().squeeze(), cmap = 'gray')
    plt.title("Ground truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pr_mask.detach().cpu().numpy())
    plt.title("Prediction")
    plt.axis("off")

    plt.show()

In [ ]:
print("Done!")
end_time = datetime.now()
print('Duration: {}'.format(end_time - start_time))

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">Part 6 — Summary & Next Steps</p> <a id="part6"></a>

### What We Covered

| Topic | Key Takeaway |
|-------|-------------|
| **Brain tumors** | Gliomas are the most common primary brain tumors; automated segmentation aids diagnosis |
| **Segmentation** | Pixel-level classification; Dice and IoU preferred over accuracy |
| **U-Net** | Encoder-decoder with skip connections; designed for biomedical segmentation |
| **PyTorch pipeline** | Custom Dataset/DataLoader, manual training loop, smp metrics |
| **SOTA models** | nnU-Net, Swin UNETR, MedSAM, TransBTS, Attention U-Net |

### Next Steps

1. **Try the TF/Keras version** — compare implementation and results (`SPARK_BrainTumorSeg2026.ipynb`)
2. **Use pre-trained encoders** — `smp.Unet(encoder_name='resnet34', encoder_weights='imagenet')`
3. **Try nnU-Net** — install and run for automated configuration
4. **3D segmentation** — Extend to BraTS dataset with multi-modal volumetric MRI
5. **Post-processing** — Connected component analysis, morphological operations
6. **Multi-class** — Segment tumor sub-regions (enhancing, core, whole)

# <p style="background-color:#1a5276;color:white;font-size:25px;text-align:center;border-radius:10px;font-weight:bold;padding:10px">References</p> <a id="refs"></a>

1. **U-Net**: Ronneberger et al., MICCAI 2015. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)
2. **nnU-Net**: Isensee et al., *Nature Methods* 2021. [DOI:10.1038/s41592-020-01008-z](https://doi.org/10.1038/s41592-020-01008-z) | [GitHub](https://github.com/MIC-DKFZ/nnUNet)
3. **Swin UNETR**: Hatamizadeh et al., BrainLes/MICCAI 2021. [arXiv:2201.01266](https://arxiv.org/abs/2201.01266) | Tang et al., CVPR 2022. [arXiv:2111.14791](https://arxiv.org/abs/2111.14791) | [GitHub](https://github.com/Project-MONAI/tutorials)
4. **MedSAM**: Ma et al., *Nature Communications* 2024. [DOI:10.1038/s41467-024-44824-z](https://doi.org/10.1038/s41467-024-44824-z) | [GitHub](https://github.com/bowang-lab/MedSAM) | [Brain](https://github.com/vpulab/med-sam-brain)
5. **TransBTS**: Wang et al., MICCAI 2021. [DOI:10.1007/978-3-030-87193-2_11](https://doi.org/10.1007/978-3-030-87193-2_11) | [GitHub](https://github.com/Wenxuan-1119/TransBTS)
6. **Attention U-Net**: Oktay et al., MIDL 2018. [arXiv:1804.03999](https://arxiv.org/abs/1804.03999)
7. **Regional-Aware U-Net**: Abbas et al. [GitHub](https://github.com/abbas695/Regional_aware_U-NET)
8. **LGG Dataset**: Buda et al., 2019. [Kaggle](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation)
9. **BraTS Challenge**: Menze et al., IEEE TMI 2015. [DOI:10.1109/TMI.2014.2377694](https://doi.org/10.1109/TMI.2014.2377694)
10. **segmentation-models-pytorch**: Yakubovskiy, P. [GitHub](https://github.com/qubvel/segmentation_models.pytorch)